# Notebook 8: Going Further

**Important**: This notebook sits outside the core 7-notebook sequence. Its goal isn't to make you fluent in these topics, but to **open the door** to them: you'll meet all three again, in much more depth, in future projects.

---

## 0. Why this notebook exists

Throughout this course, we deliberately kept two things out of scope: proper error handling, and object-oriented programming. That was intentional as they weren't essential to get you writing useful Python. But you'll run into both very soon (pandas and scikit-learn are full of objects, and real programs need to handle failure gracefully), so it's worth a first, gentle look now.

We'll cover three things:
1. **Exceptions**: handling errors instead of letting them crash your program.
2. **Classes**: a different way of organizing code, bundling data and behavior together.
3. **Duck typing**: a very Python-specific idea about *how* types work, with real trade-offs.

---

## 1. Exceptions

You've already seen Python raise errors like `IndexError`, `KeyError`, `TypeError`, and so on. Until now, when one of these happened, the program just stopped. **Exception handling** lets you catch an error and decide what to do instead of crashing.

In [ ]:
numbers = [1, 2, 3]

try:
    print(numbers[10])
except IndexError:
    print("That index doesn't exist in the list.")

print("The program keeps running after this.")

The `try` block contains code that might fail. If an exception matching the `except` clause is raised, Python jumps straight to that block instead of crashing and everything else keeps running.

You can catch different exception types differently, and catch several types at once:

In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Cannot divide by zero.")
        return None
    except TypeError:
        print("Both arguments must be numbers.")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(10, "a"))

Two extra keywords round out the pattern:
- `else`: runs only if the `try` block succeeded (no exception).
- `finally`: always runs, whether there was an exception or not (useful for cleanup, like closing a file).

In [ ]:
def load_config(filename):
    try:
        f = open(filename)
    except FileNotFoundError:
        print(f"{filename} not found, using defaults.")
        return {}
    else:
        content = f.read()
        print("File loaded successfully.")
        return {"raw": content}
    finally:
        print("Done attempting to load the config.")

load_config("does_not_exist.txt")

Finally, you can raise your own exceptions with `raise`. This is useful when your own code detects an invalid situation:

In [ ]:
def set_age(age):
    if age < 0:
        raise ValueError("Age cannot be negative.")
    return age

try:
    set_age(-5)
except ValueError as e:
    print("Caught an error:", e)

This "try it, handle failure if it happens" style is called **EAFP** ("Easier to Ask Forgiveness than Permission") and it's often preferred in Python over checking every precondition manually before acting.

### Exercise 1: Exceptions
Write a function `safe_get(dictionary, key)` that returns `dictionary[key]` if the key exists, or `None` (after printing a message) if it doesn't, using `try`/`except KeyError`, not an `if`/`in` check.

In [ ]:
def safe_get(dictionary, key):
    # your code here
    pass

teacher = {"name": "William", "age": 29}
print(safe_get(student, "name"))
print(safe_get(student, "city"))


---

## 2. Classes: a different way to organize code

Everything you've written in this course is **procedural**: data (variables, lists, dicts, DataFrames...) on one side, functions that operate on that data on the other. It's simple and it scales surprisingly far: plenty of real data science code stays procedural throughout.

**Object-oriented programming (OOP)** takes a different approach: it bundles data and the functions that operate on it together, into a single unit called an **object**. A **class** is the blueprint for creating such objects.

Let's compare the two styles on the same problem: representing a student and computing their grade average.

In [ ]:
# Procedural style: data and functions are separate
student = {"name": "Alice", "grades": [15, 17, 13]}

def average_grade(student):
    return sum(student["grades"]) / len(student["grades"])

print(average_grade(student))

In [ ]:
# Object-oriented style: data and behavior live together
class Student:
    def __init__(self, name, grades):
        self.name = name       # these are the object's data ("attributes")
        self.grades = grades

    def average_grade(self):   # this is the object's behavior (a "method")
        return sum(self.grades) / len(self.grades)


bob = Student("Bob", [13, 12, 20])   # creates an "instance" of the class
print(bob.name)
print(bob.average_grade())

A few things to notice:
- `__init__` is a special method that runs when you create a new object (`Student("Bob", [13, 12, 20])`): it sets up the initial data.
- `self` refers to "this particular object": it's how a method accesses its own data.
- Calling `bob.average_grade()` reads almost like a sentence: "ask Bob for their average grade," rather than "pass Bob's data into a function."

### Why (and when) does this matter?

OOP tends to pay off when:
- You have many objects of the same "shape" that carry both data *and* behavior specific to them (e.g. many `Student` objects, each able to compute their own results).
- You want to group related data and operations together so they can't drift apart, or be misused with the wrong function.
- You're building something with a long-lived, evolving state (a UI element, a user profile, a machine learning model object $\Rightarrow$ scikit-learn's models are actually classes, with methods like `.fit()` and `.predict()`).

Procedural code tends to stay simpler and more appropriate when:
- You're mostly transforming data through a pipeline of steps, which is most of what you're going to do in this course (and much of what data analysis looks like).
- There isn't really a natural "object" with its own ongoing state, just data flowing through functions.

Neither style is objectively "better": they're different tools. Python happily lets you mix both, and you'll do so constantly: you'll write your own functions (procedural) that call methods on objects from libraries (OOP). A pandas DataFrame, for instance, is itself an object with methods like `.head()` and `.groupby()`.

### Exercise 2: A first class
Write a class `Rectangle` with `__init__(self, width, height)` and a method `area(self)` that returns `width * height`. Create an instance with width `4` and height `7`, and print its area.

In [ ]:
class Rectangle:
    # your code here
    pass

---

## 3. Duck typing

Python is a **dynamically typed** language: you don't declare a variable's type ahead of time, and the same function can happily accept very different types of objects, as long as they support whatever operations the function actually uses. This idea is called **duck typing**: "if it walks like a duck and quacks like a duck, it's a duck."

In [ ]:
def describe_length(item):
    return f"This has {len(item)} elements."

print(describe_length([1, 2, 3]))          # a list
print(describe_length("hello"))            # a string
print(describe_length({"a": 1, "b": 2}))   # a dictionary

`describe_length` never checks *what type* `item` is, it just assumes `item` supports `len()`, and calls it. A list, a string, and a dictionary are all completely different types, but they all "quack like a duck", so the same function works on all of them.

This connects directly to the `try`/`except` idea from earlier: rather than checking a type *before* acting (which duck typing tends to avoid), idiomatic Python often just tries the operation and catches the exception if the object turns out not to support it.

In [ ]:
def add_prefix(item, prefix):
    try:
        return prefix + item
    except TypeError:
        return f"Can't add a prefix to a {type(item).__name__}."

print(add_prefix("world", "hello "))
print(add_prefix(1234, "hello "))

### Advantages

- **Flexibility**: one function can work with many unrelated types, as long as they support the right operations: no need to write a separate version for each type, or force everything into a shared class hierarchy.
- **Less upfront ceremony**: you don't need to design and declare formal type relationships before writing code that works: you can write a function around the behavior you need immediately.

### Problems

- **Errors surface late**: if you pass something incompatible, you often only find out when the function actually tries (and fails at) the unsupported operation $\rightarrow$ possibly deep inside a large program, far from where the mistake was made.
- **Contracts are implicit**: nothing in `describe_length(item)`'s signature tells you `item` needs to support `len()`, you have to read the function body (or its docstring, if someone bothered to write one) to know what's actually expected. Type hints help mitigate this, but don't fully solve it, since they aren't enforced.
- **Silent wrong behavior is possible, not just crashes**: the risk isn't only exceptions, sometimes an object "quacks" just enough to be accepted, but behaves subtly differently than intended, producing a wrong result without ever raising an error.

Duck typing is a genuine design trade-off, not a flaw to avoid. It's a big part of why Python code often feels quick to write and flexible to extend. But it's worth knowing it's a deliberate choice, with a real cost in how early (and how loudly) mistakes tend to surface.

### Exercise 3: Duck typing
Write a function `total_items(collection)` that returns the sum of a numeric collection using `sum()`, without checking its type. Test it on a list `[1, 2, 3]`, a tuple `(4, 5, 6)`, and a set `{7, 8, 9}`. The same function should work on all three, because they all "quack" the same way for `sum()`.

In [ ]:
def total_items(collection):
    # your code here
    pass

print(total_items([1, 2, 3]))
print(total_items((4, 5, 6)))
print(total_items({7, 8, 9}))

---

## Wrap-up

This supplement opened three doors without walking all the way through any of them:
- **Exceptions**: `try`/`except`/`else`/`finally`, raising your own errors with `raise`, and the EAFP mindset of "try it, handle failure if it happens"
- **Classes**: bundling data and behavior into objects, `__init__`/`self`/methods, and when OOP tends to pay off compared to the procedural style used throughout this course
- **Duck typing**: Python's "if it behaves like the right type, it *is* the right type" philosophy, its flexibility, and its cost in how late mistakes can surface

You'll meet all three again, in real depth, as you go further. Exceptions and classes are everywhere in production code, and duck typing is quietly shaping how most Python libraries (including pandas and scikit-learn) are designed to be used.